In [9]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install datasets transformers tokenizers sentencepiece tqdm accelerate -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [10]:
# ============================================================
# CELL 2 — Imports & Config
# ============================================================
import torch
import re, math, os
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, IterableDataset
from tqdm import tqdm
from datasets import load_dataset, interleave_datasets
from transformers import LlamaTokenizer
import sentencepiece as spm
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# UPDATED CONFIG FOR 2GB VRAM
# ============================================================
config = {
    "num_samples": 1_000_000,
    "vocab_train_size": 100_000,
    "max_len": 64,          # Reduced sequence length for VRAM
    "vocab_size": 32_000,
    "n_layers": 6,          # Reduced layers (originally 12)
    "hidden_size": 256,     # Reduced hidden size (originally 768)
    "n_heads": 8,           # Adjusted for hidden_size
    "dropout": 0.1,
    "batch_size": 16,       # Lower batch size for 2GB VRAM
    "grad_accum_steps": 16, # Increased to keep effective batch size stable
    "lr": 5e-5,
    "weight_decay": 0.01,
    "warmup_steps": 3000,
    "max_steps": 50000,
    "save_every": 5000,
    "log_every": 100,
    "spm_prefix": "indic_gpt_pro",
    "checkpoint_dir": "checkpoints",
}
os.makedirs(config["checkpoint_dir"], exist_ok=True)
print(f"Config ready. Using device: {device}")

Config ready. Using device: cpu


In [11]:
# ============================================================
# CELL 3 — Dataset
# ============================================================
def clean_text(example):
    # Samanantar uses 'src' and 'tgt'. We merge them for LM training.
    text = f"{example['src']} {example['tgt']}"
    text = re.sub(r"\s+", " ", text).strip()
    # Return None if text is too short; we will filter these out
    return {"text": text if len(text) > 10 else None}

print("Streaming datasets...")
ds_hi = load_dataset("ai4bharat/samanantar", "hi", split="train", streaming=True).take(config["num_samples"])
ds_bn = load_dataset("ai4bharat/samanantar", "bn", split="train", streaming=True).take(config["num_samples"])

dataset = interleave_datasets([ds_hi, ds_bn], stopping_strategy="all_exhausted", seed=42).map(clean_text)
dataset = dataset.filter(lambda x: x["text"] is not None)
print("Dataset pipeline ready ✓")

Streaming datasets...
Dataset pipeline ready ✓


In [12]:
import sentencepiece as spm
from transformers import LlamaTokenizer
import os
import json
import shutil

spm_model_path = f"{config['spm_prefix']}.model"
spm_vocab_path = f"{config['spm_prefix']}.vocab"

if not os.path.exists(spm_model_path):
    print("Training SentencePiece...")
    with open("vocab_train.txt", "w", encoding="utf-8") as f:
        for example in dataset.take(config["vocab_train_size"]):
            f.write(example["text"] + "\n")
    spm.SentencePieceTrainer.train(
        input="vocab_train.txt", model_prefix=config["spm_prefix"],
        vocab_size=config["vocab_size"], model_type="unigram",
        byte_fallback=True, pad_id=0, unk_id=1, bos_id=2, eos_id=3
    )

# --- Diagnostic: Verify SentencePiece model directly ---
sp_processor = spm.SentencePieceProcessor()
sp_processor.load(spm_model_path)
print(f"SentencePiece Processor directly reports vocab size: {sp_processor.get_piece_size()}")

# Create a temporary directory for the tokenizer files
tokenizer_dir = "temp_tokenizer_output"
os.makedirs(tokenizer_dir, exist_ok=True)

# Copy the trained SentencePiece model to the directory with the expected name
shutil.copy(spm_model_path, os.path.join(tokenizer_dir, "tokenizer.model"))

# Create tokenizer_config.json
tokenizer_config_data = {
    "model_max_length": config["max_len"],
    "bos_token": "<s>",
    "eos_token": "</s>",
    "unk_token": "<unk>",
    "pad_token": "<pad>",
    "add_prefix_space": False,
    "clean_up_tokenization_spaces": True,
    "legacy": False,
    "tokenizer_class": "LlamaTokenizer",
    "pad_token_id": sp_processor.pad_id(),
    "unk_token_id": sp_processor.unk_id(),
    "bos_token_id": sp_processor.bos_id(),
    "eos_token_id": sp_processor.eos_id()
}
with open(os.path.join(tokenizer_dir, "tokenizer_config.json"), "w") as f:
    json.dump(tokenizer_config_data, f, indent=2)

# Create special_tokens_map.json
special_tokens_map_data = {
    "pad_token": "<pad>",
    "bos_token": "<s>",
    "eos_token": "</s>",
    "unk_token": "<unk>"
}
with open(os.path.join(tokenizer_dir, "special_tokens_map.json"), "w") as f:
    json.dump(special_tokens_map_data, f, indent=2)

# Load the LlamaTokenizer from the created directory
tokenizer = LlamaTokenizer.from_pretrained(tokenizer_dir)

print(f"Tokenizer ready | Vocab size: {len(tokenizer)}")
print(f"Pad token ID: {tokenizer.pad_token_id}, Unk token ID: {tokenizer.unk_token_id}, Bos token ID: {tokenizer.bos_token_id}, Eos token ID: {tokenizer.eos_token_id}")

SentencePiece Processor directly reports vocab size: 16000
Tokenizer ready | Vocab size: 16000
Pad token ID: 0, Unk token ID: 1, Bos token ID: 2, Eos token ID: 3


In [13]:
# ============================================================
# CELL 5 — Model Architecture (FIXED)
# ============================================================
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["hidden_size"])
        self.pos_emb = nn.Embedding(cfg["max_len"], cfg["hidden_size"])
        self.drop = nn.Dropout(cfg["dropout"])

        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=cfg["hidden_size"],
                nhead=cfg["n_heads"],
                dim_feedforward=4*cfg["hidden_size"],
                dropout=cfg["dropout"],
                activation="gelu",
                batch_first=True,
                norm_first=True
            ) for _ in range(cfg["n_layers"])
        ])

        self.ln_f = nn.LayerNorm(cfg["hidden_size"])
        self.lm_head = nn.Linear(cfg["hidden_size"], cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.token_emb.weight

        # Apply manual stable initialization
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.drop(self.token_emb(x) + self.pos_emb(pos))
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        for block in self.blocks:
            h = block(h, src_mask=mask)

        return self.lm_head(self.ln_f(h))

model = GPTModel(config).to(device)
print("Model re-initialized with stable weights ✓")

Model re-initialized with stable weights ✓


In [14]:
# ============================================================
# CELL 6 — Streaming Dataset + DataLoader
# ============================================================

class StreamingTextDataset(IterableDataset):
    def __init__(self, hf_dataset, tokenizer, max_len):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __iter__(self):
        for example in self.dataset:
            text = example.get("text", "")
            if not text or len(text.strip()) < 5:  # Skip empty or very short lines
                continue

            enc = self.tokenizer(text, truncation=True, padding="max_length",
                                 max_length=self.max_len, return_tensors="pt")
            yield enc["input_ids"].squeeze(0)

train_loader = DataLoader(StreamingTextDataset(dataset, tokenizer, config["max_len"]), batch_size=config["batch_size"])
print("DataLoader ready with data guards ✓")

DataLoader ready with data guards ✓


In [15]:
# ============================================================
# CELL 7 — Sanity Check (Run Before Training!)
# ============================================================
print("Running Sanity Check...")
test_batch = next(iter(train_loader)).to(device)
with torch.no_grad():
    logits = model(test_batch[:, :-1])
print(f"Batch Shape: {test_batch.shape} | Logits Shape: {logits.shape}")
print("Sanity check passed! Safe to train.")

Running Sanity Check...
Batch Shape: torch.Size([16, 64]) | Logits Shape: torch.Size([16, 63, 32000])
Sanity check passed! Safe to train.


In [16]:
# ============================================================
# CELL 8 — Optimizer & Graphing Function
# ============================================================

optimizer = AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
scaler = torch.cuda.amp.GradScaler()
history = {"step": [], "loss": [], "lr": []} # Clears the NaN history

def plot_metrics(hist):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(hist["step"], hist["loss"], color='blue')
    ax1.set_title("Training Loss")
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss")
    ax1.grid(True)

    ax2.plot(hist["step"], hist["lr"], color='green')
    ax2.set_title("Learning Rate")
    ax2.set_xlabel("Step")
    ax2.grid(True)
    plt.show()

print("Optimizer and history reset ✓")

Optimizer and history reset ✓


C:\Users\vicky\AppData\Local\Temp\ipykernel_15632\2182232013.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
# ============================================================
# CELL 9 — Training Loop (LIVE UPDATES - NO VAR CHANGES)
# ============================================================
model.train()
step, accum_loss = 0, 0
pbar = tqdm(total=config["max_steps"])

for batch in train_loader:
    if step >= config["max_steps"]: break
    inputs = batch.to(device)

    # Linear Warmup based on current config
    curr_lr = config["lr"] * min(1.0, step / max(1, config["warmup_steps"]))
    for pg in optimizer.param_groups: pg["lr"] = curr_lr

    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        logits = model(inputs[:, :-1])
        # Stability fix: Compute loss in float32
        loss = F.cross_entropy(
            logits.float().reshape(-1, logits.size(-1)),
            inputs[:, 1:].reshape(-1),
            ignore_index=tokenizer.pad_token_id
        ) / config["grad_accum_steps"]

    # Check for NaN
    if torch.isnan(loss):
        print(f"\n⚠️ NaN detected at step {step}. Skipping batch...")
        optimizer.zero_grad()
        continue

    scaler.scale(loss).backward()
    accum_loss += loss.item()

    # --- LIVE UPDATE SECTION ---
    # We show the loss for the current batch immediately on the progress bar
    live_loss = loss.item() * config["grad_accum_steps"]
    pbar.set_postfix(loss=f'{live_loss:.4f}', lr=f'{curr_lr:.6f}')
    pbar.update(1)
    # ---------------------------

    if (step + 1) % config["grad_accum_steps"] == 0:
        scaler.unscale_(optimizer)
        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        if step % config["log_every"] == 0:
            history["step"].append(step)
            avg_loss_for_logging = accum_loss * (1.0 / (config["grad_accum_steps"] if config["grad_accum_steps"] > 0 else 1))
            # Note: keeping your logic of multiplying back for the history list
            history["loss"].append(accum_loss)
            history["lr"].append(curr_lr)
            plot_metrics(history)
            accum_loss = 0

    step += 1

pbar.close()


  0%|                                                 | 12/25000 [04:12<146:00:26, 21.04s/it, loss=9.7692, lr=0.000000]

  0%|                                                 | 53/50000 [01:29<19:37:25,  1.41s/it, loss=10.4233, lr=0.000001]

In [ ]:
model_save_path = os.path.join(config["checkpoint_dir"], "final_model.pt")
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
# ============================================================
# CELL 10 — Generation
# ============================================================

@torch.inference_mode()
def generate(prompt, max_new_tokens=50):
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    for _ in range(max_new_tokens):
        logits = model(ids[:, -config["max_len"]:])[:, -1, :]
        next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id: break
    return tokenizer.decode(ids[0], skip_special_tokens=True)

print(generate("भारतीय अर्थव्यवस्था"))

In [ ]:
# ============================================================
# NEW SECTION: Post-Training Analysis & Divergence Visualization
# ============================================================
import numpy as np

def plot_final_analysis(hist):
    if not hist["loss"]:
        print("No history found to plot.")
        return

    steps = hist["step"]
    loss_values = hist["loss"]
    
    plt.figure(figsize=(12, 6))
    
    # 1. Training Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(steps, loss_values, label='Training Loss', color='#1f77b4')
    
    # Calculate a moving average to see the trend (Divergence Check)
    window = 10
    if len(loss_values) > window:
        mov_avg = np.convolve(loss_values, np.ones(window)/window, mode='valid')
        plt.plot(steps[window-1:], mov_avg, label='Trend (Moving Avg)', color='red', linestyle='--')
    
    plt.title("Loss Convergence & Stability")
    plt.xlabel("Steps")
    plt.ylabel("Cross Entropy Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # 2. Loss Derivative (Rate of Change)
    # This helps identify if the model is diverging or plateauing
    plt.subplot(1, 2, 2)
    gradients = np.diff(loss_values)
    plt.plot(steps[1:], gradients, color='purple', alpha=0.5)
    plt.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    plt.title("Loss Gradient (Divergence Indicator)")
    plt.xlabel("Steps")
    plt.ylabel("Change in Loss")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

    # Summary Statistics
    print(f"Initial Loss: {loss_values[0]:.4f}")
    print(f"Final Loss: {loss_values[-1]:.4f}")
    print(f"Total Reduction: {((loss_values[0] - loss_values[-1]) / loss_values[0]) * 100:.2f}%")

plot_final_analysis(history)